In [ ]:
from google.colab import files
files.upload()

In [ ]:
!pip install catboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.formula.api as smf
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB,MultinomialNB,BernoulliNB
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import VotingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from catboost import CatBoostClassifier

In [ ]:
df_df = pd.read_csv('Data_of_Attack_Back.csv')

In [ ]:
df_df.head()

In [ ]:
df_df.head()

In [ ]:
df_buffer = pd.read_csv('Data_of_Attack_Back_BufferOverflow.csv')

In [ ]:
df_buffer.head()

In [ ]:
df_columns = df_buffer.columns

In [ ]:
df_columns

In [ ]:
df_ftpwrite = pd.read_csv('Data_of_Attack_Back_FTPWrite.csv',names=df_columns)

In [ ]:
df_ftpwrite.head()

In [ ]:
df_guesspass = pd.read_csv('Data_of_Attack_Back_GuessPassword.csv')

In [ ]:
df_guesspass.head()

In [ ]:
df_nmap = pd.read_csv('Data_of_Attack_Back_NMap.csv')

In [ ]:
df_nmap.head()

In [ ]:
df_neptune = pd.read_csv('Data_of_Attack_Back_Neptune.csv')

In [ ]:
df_neptune.head()

In [ ]:
df_normal = pd.read_csv('Data_of_Attack_Back_Normal.csv')

In [ ]:
df_normal.head()

In [ ]:
df_portsweep = pd.read_csv('Data_of_Attack_Back_PortSweep.csv')

In [ ]:
df_portsweep.head()

In [ ]:
df_rootkit = pd.read_csv('Data_of_Attack_Back_RootKit.csv')

In [ ]:
df_rootkit.head()

In [ ]:
df_satan = pd.read_csv('Data_of_Attack_Back_Satan.csv')

In [ ]:
df_satan.head()

In [ ]:
df_smurf = pd.read_csv('Data_of_Attack_Back_Smurf.csv')

In [ ]:
df_smurf.head()

In [ ]:
df_df['attack_type'] = 'back'
df_buffer['attack_type'] = 'buffer_overflow'
df_ftpwrite['attack_type'] = 'ftp_write'
df_guesspass['attack_type'] = 'guess_password'
df_nmap['attack_type'] = 'nmap'
df_neptune['attack_type'] = 'neptune'
df_normal['attack_type'] = 'normal'
df_portsweep['attack_type'] = 'portsweep'
df_rootkit['attack_type'] = 'rootkit'
df_satan['attack_type'] = 'satan'
df_smurf['attack_type'] = 'smurf'


In [ ]:
df_full = pd.concat([df_df, df_buffer, df_ftpwrite, df_guesspass,
                     df_nmap, df_neptune, df_normal, df_portsweep,
                     df_rootkit, df_satan, df_smurf], axis=0, ignore_index=True)

In [ ]:
df_full.columns = df_full.columns.str.strip()

In [ ]:
df_full.columns

In [ ]:
df_full.info()

In [ ]:
df_full.shape
df_full['attack_type'].value_counts()

In [ ]:
df_full['attack_type_binary'] = np.where(df_full['attack_type'] == 'normal','normal',
                                  'attack')

In [ ]:
df_full.columns

In [ ]:
df_full['attack_type_binary'] = np.where(df_full['attack_type_binary']=='attack',1,0)

In [ ]:
df_full['attack_type_binary'].value_counts()

In [ ]:
label = LabelEncoder()

In [ ]:
df_full['attack_type_multi'] = label.fit_transform(df_full['attack_type'])

In [ ]:
df_full.columns

In [ ]:
df_full.info()

In [ ]:
sns.boxplot(df_full['attack_type_multi'])

In [ ]:

sns.distplot(df_full['attack_type_multi'].head(980))

In [ ]:
df = df_full.select_dtypes(include=['float', 'int'])

In [ ]:
df.info()

In [ ]:
def outliers_influence(x):
  x = x.clip(upper= x.quantile(0.99),lower= x.quantile(0.01))
  return x

In [ ]:
df = df.apply(outliers_influence)

In [ ]:
df.info()

In [ ]:
sns.boxplot(df['attack_type_multi'])

In [ ]:
sns.distplot(df['attack_type_multi'].head(1070))

In [ ]:
sel_col = df.drop(columns=['attack_type_multi','attack_type_binary'],axis=1)

In [ ]:
df.columns

In [ ]:
df['is_guest_login'].value_counts()

In [ ]:
vif = pd.DataFrame()
vif['feature'] = sel_col.columns
vif['VIF'] = [variance_inflation_factor(sel_col.values,i) for i in range(sel_col.shape[1])]

In [ ]:
vif

In [ ]:
vif_sel = vif.loc[vif['VIF']>10]

In [ ]:
vif_sel['feature'].values

In [ ]:
df_sel = df[['flag', 'logged_in', 'serror_rate', 'srv_error_rate',
       'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_serror_rate', 'dst_host_srv_serror_rate',
       'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
             'attack_type_multi','attack_type_binary']]

In [ ]:
x = df_sel.drop(['attack_type_multi','attack_type_binary'],axis=1)
y_multi = df_sel[['attack_type_multi']]

In [ ]:
x.columns

In [ ]:
y_multi.columns

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y_multi,test_size=0.2,random_state=123)

In [ ]:
std = StandardScaler()

In [ ]:
x_train_std = std.fit_transform(x_train)
x_test_std = std.transform(x_test)

In [ ]:
df_sel.columns

In [ ]:
KNeighborsClassifier?

In [ ]:
para_knn = {}

In [ ]:
GridSearchCV?

In [ ]:
grid_cv_knn = GridSearchCV(KNeighborsClassifier(),param_grid=para_knn,cv=2)

In [ ]:
grid_cv_knn = grid_cv_knn.fit(x_train_std,y_train)

In [ ]:
grid_cv_knn.best_score_

In [ ]:
model_knn = grid_cv_knn.best_estimator_

In [ ]:
model_knn = model_knn.fit(x_train_std, y_train)

In [ ]:
print(classification_report(y_train, model_knn.predict(x_train_std)))

In [ ]:
print(classification_report(y_test, model_knn.predict(x_test_std)))

In [ ]:
GaussianNB?

In [ ]:
para_gaussian = {}

In [ ]:
grid_cv_gaussian = GridSearchCV(GaussianNB(),param_grid=para_gaussian,cv=2)

In [ ]:
grid_cv_gaussian = grid_cv_gaussian.fit(x_train,y_train)

In [ ]:
grid_cv_gaussian.best_score_

In [ ]:
model_gaussian = grid_cv_gaussian.best_estimator_

In [ ]:
model_gaussian = model_gaussian.fit(x_train,y_train)

In [ ]:
print(classification_report(y_train, model_gaussian.predict(x_train)))

In [ ]:
print(classification_report(y_test, model_gaussian.predict(x_test)))

In [ ]:
DecisionTreeClassifier?

In [ ]:
para_tree = {}

In [ ]:
grid_cv_tree = GridSearchCV(DecisionTreeClassifier(),param_grid=para_tree,cv=2)

In [ ]:
grid_cv_tree = grid_cv_tree.fit(x_train,y_train)

In [ ]:
grid_cv_tree.best_score_

In [ ]:
model_tree = grid_cv_tree.best_estimator_

In [ ]:
model_tree = model_tree.fit(x_train,y_train)

In [ ]:
print(classification_report(y_train, model_tree.predict(x_train)))

In [ ]:
print(classification_report(y_test, model_tree.predict(x_test)))

In [ ]:
BaggingClassifier?

In [ ]:
para_bag = {}

In [ ]:
grid_cv_bag = GridSearchCV(BaggingClassifier(),param_grid=para_bag,cv=2)

In [ ]:
grid_cv_bag = grid_cv_bag.fit(x_train,y_train)

In [ ]:
grid_cv_bag.best_score_

In [ ]:
model_bag = grid_cv_bag.best_estimator_

In [ ]:
model_bag = model_bag.fit(x_train,y_train)

In [ ]:
print(classification_report(y_train, model_bag.predict(x_train)))

In [ ]:
print(classification_report(y_test, model_bag.predict(x_test)))

In [ ]:
RandomForestClassifier?

In [ ]:
para_forest = {}

In [ ]:
grid_cv_forest = GridSearchCV(RandomForestClassifier(),param_grid=para_forest,cv=2)

In [ ]:
grid_cv_forest = grid_cv_forest.fit(x_train,y_train)

In [ ]:
grid_cv_forest.best_score_

In [ ]:
model_forest = grid_cv_forest.best_estimator_

In [ ]:
model_forest = model_forest.fit(x_train,y_train)

In [ ]:
print(classification_report(y_train, model_forest.predict(x_train)))

In [ ]:
print(classification_report(y_test, model_forest.predict(x_test)))

In [ ]:
AdaBoostClassifier?

In [ ]:
para_ada = {}

In [ ]:
grid_cv_ada = GridSearchCV(AdaBoostClassifier(),param_grid=para_ada,cv=2)

In [ ]:
grid_cv_ada = grid_cv_ada.fit(x_train,y_train)

In [ ]:
grid_cv_ada.best_score_

In [ ]:
model_ada = grid_cv_ada.best_estimator_

In [ ]:
model_ada = model_ada.fit(x_train,y_train)

In [ ]:
print(classification_report(y_train, model_ada.predict(x_train)))

In [ ]:
print(classification_report(y_test, model_ada.predict(x_test)))

In [ ]:
GradientBoostingClassifier?

In [ ]:
para_gradient = {}

In [ ]:
grid_cv_gradient = GridSearchCV(GradientBoostingClassifier(),param_grid=para_gradient,cv=2)

In [ ]:
grid_cv_gradient = grid_cv_gradient.fit(x_train,y_train)

In [ ]:
grid_cv_gradient.best_score_

In [ ]:
model_gradient = grid_cv_gradient.best_estimator_

In [ ]:
model_gradient = model_gradient.fit(x_train,y_train)

In [ ]:
print(classification_report(y_train, model_gradient.predict(x_train)))

In [ ]:
print(classification_report(y_test, model_gradient.predict(x_test)))

In [ ]:
XGBClassifier?

In [ ]:
para_xgb = {}

In [ ]:
grid_cv_xgb = GridSearchCV(XGBClassifier(),param_grid=para_xgb,cv=2)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_xgb = LabelEncoder()
# Flatten y_train DataFrame to a 1D array before encoding
y_train_encoded_for_xgb = le_xgb.fit_transform(y_train.squeeze())
y_test_encoded_for_xgb = le_xgb.transform(y_test.squeeze())

grid_cv_xgb = grid_cv_xgb.fit(x_train, y_train_encoded_for_xgb)

In [ ]:
grid_cv_xgb.best_score_

In [ ]:
model_xgb = grid_cv_xgb.best_estimator_

In [ ]:
model_xgb = model_xgb.fit(x_train,y_train_encoded_for_xgb)

In [ ]:
print(classification_report(y_train_encoded_for_xgb, model_xgb.predict(x_train)))

In [ ]:
print(classification_report(y_test_encoded_for_xgb, model_xgb.predict(x_test)))

In [ ]:
LGBMClassifier?

In [ ]:
para_lgbm = {}

In [ ]:
grid_cv_lgbm = GridSearchCV(LGBMClassifier(),param_grid=para_lgbm,cv=2)

In [ ]:
grid_cv_lgbm = grid_cv_lgbm.fit(x_train,y_train)

In [ ]:
grid_cv_lgbm.best_score_

In [ ]:
model_lgbm = grid_cv_lgbm.best_estimator_

In [ ]:
model_lgbm = model_lgbm.fit(x_train,y_train)

In [ ]:
print(classification_report(y_train, model_lgbm.predict(x_train)))

In [ ]:
print(classification_report(y_test, model_lgbm.predict(x_test)))

In [ ]:
CatBoostClassifier?

In [ ]:
para_cat = {}

In [ ]:
grid_cv_cat = GridSearchCV(CatBoostClassifier(),param_grid=para_cat,cv=2)

In [ ]:
grid_cv_cat = grid_cv_cat.fit(x_train,y_train)

In [ ]:
grid_cv_cat.best_score_

In [ ]:
model_cat = grid_cv_cat.best_estimator_

In [ ]:
model_cat = model_cat.fit(x_train,y_train)

In [ ]:
print(classification_report(y_train, model_cat.predict(x_train)))

In [ ]:
print(classification_report(y_test, model_cat.predict(x_test)))

In [ ]:
SVC?

In [ ]:
para_svc = {}

In [ ]:
grid_cv_svc = GridSearchCV(SVC(),param_grid=para_svc,cv=2)

In [ ]:
grid_cv_svc = grid_cv_svc.fit(x_train_std,y_train)

In [ ]:
grid_cv_svc.best_score_

In [ ]:
model_svc = grid_cv_svc.best_estimator_

In [ ]:
model_svc = model_svc.fit(x_train_std,y_train)

In [ ]:
print(classification_report(y_train, model_svc.predict(x_train_std)))

In [ ]:
print(classification_report(y_test, model_svc.predict(x_test_std)))

In [ ]:
VotingClassifier?

In [ ]:
estimator = []

In [ ]:
estimator.append(('knn',grid_cv_knn.best_estimator_))
estimator.append(('gaussian',grid_cv_gaussian.best_estimator_))
estimator.append(('tree',grid_cv_tree.best_estimator_))
estimator.append(('bag',grid_cv_bag.best_estimator_))
estimator.append(('forest',grid_cv_forest.best_estimator_))
estimator.append(('ada',grid_cv_ada.best_estimator_))
estimator.append(('gradient',grid_cv_gradient.best_estimator_))
estimator.append(('xgb',grid_cv_xgb.best_estimator_))
estimator.append(('lgbm',grid_cv_lgbm.best_estimator_))
estimator.append(('cat',grid_cv_cat.best_estimator_))
estimator.append(('svc',grid_cv_svc.best_estimator_))


In [ ]:
para_vote = {}

In [ ]:
grid_cv_vote = GridSearchCV(VotingClassifier(estimators=estimator),param_grid=para_vote,cv=2)

In [ ]:
grid_cv_vote = grid_cv_vote.fit(x_train,y_train)

In [ ]:
grid_cv_vote.best_score_

In [ ]:
model_vote = grid_cv_vote.best_estimator_

In [ ]:
model_vote = model_vote.fit(x_train,y_train)

In [ ]:
best_score = pd.DataFrame({
    'knn': 0.9939009846492569,
    'gaussian': 0.9798850223227937,
    'tree': 0.9941716102990643,
    'bag': 0.994211363219375,
    'forest': 0.9943290930218336,
    'ada': 0.9911610910647667,
    'gradient': 0.99313956332946,
    'xgb': 0.9946746376368418,
    'lgbm': 0.9945171549140726,
    'cat': 0.9942878111430493,
    'svc': 0.9924041343037123,
    'vote': 0.9944758730352883
}, index=[0]).T.reset_index()

In [ ]:
best_score.columns = ['model','scores']

In [ ]:
best_score.sort_values(by='scores',ascending=False).head(1)

In [ ]:
import joblib

In [ ]:
joblib.dump(grid_cv_xgb,'xgb.pkl')

In [ ]:
joblib.load('xgb.pkl')